# AwA2 CLIP feature extraction on Colab (GPU)

Builds the frozen artifacts the AwA2 transfer study ranks against — from the raw
Animals with Attributes 2 dataset:

- `artifacts/awa2/clip_image_features_train.pt` — `[N_train, 512]` L2-normalized CLIP vectors
- `artifacts/awa2/clip_image_features_test.pt`  — `[N_test, 512]` L2-normalized CLIP vectors
- `artifacts/awa2/clip_attr_prompt_bank.pt`     — `[85, n, 512]` per-predicate prompt bank (CLAY)
- plus the small metadata caches (`image_index.json`, splits, class/predicate names)

Both the **train and test** feature tables are extracted here. Only these tables + the
prompt bank + the small json/pt caches are needed downstream — **not** the 13GB of images.
So you extract here once, then copy these few hundred MB to your VM / local repo.

**You need one thing in Google Drive before running:**
1. `AwA2-data.zip` — the zipped `Animals_with_Attributes2` dataset folder
   (with `classes.txt`, `predicates.txt`, `predicate-matrix-binary.txt`, `JPEGImages/`, ...).

Edit the paths in the **Config** cell, then `Runtime → Run all`.
Total time on a T4 GPU: ~8–12 minutes for ~37k images.

## 0. Config — edit these lines

In [ ]:
# Public HTTPS URL of your repo (scripts only — the dataset is gitignored, Drive supplies it).
REPO_URL = "https://github.com/davideCabitz/Deep-Learning-Project.git"

# Where the dataset zip lives in your mounted Google Drive. This is the zipped
# Animals_with_Attributes2 folder (images + classes.txt/predicates.txt/matrix).
AWA2_ZIP = "/content/drive/MyDrive/dlproject/AwA2-data.zip"

# Where to copy the finished artifacts in Drive (so they survive the session).
DRIVE_OUT_DIR = "/content/drive/MyDrive/dlproject/awa2_artifacts"

## 1. Confirm the GPU is on

If this prints `False`, go to **Runtime → Change runtime type → T4 GPU → Save**, then rerun.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️  No GPU — enable it via Runtime → Change runtime type → T4 GPU.")

## 2. Get the project scripts

Clones the repo (scripts only). If the repo is **private**, this will fail — in that case
drag-drop the `src/awa2/` folder into the file panel on the left instead, then skip this
cell and set `%cd` to wherever you put the repo.

In [ ]:
import os
if not os.path.isdir("Deep-Learning-Project"):
    !git clone $REPO_URL
%cd Deep-Learning-Project
!ls src/awa2

## 3. Install CLIP dependency

Colab has torch; the extraction scripts also need `transformers` for CLIP ViT-B/32.

In [ ]:
!pip install -q transformers

## 4. Mount Drive and unpack the dataset

Unzips your `AwA2-data.zip` and normalizes it to `<repo>/Animals_with_Attributes2/...`
(works whether the zip holds the folder or just its contents). The `ls` at the end is your
checkpoint — you **must** see `classes.txt`, `predicates.txt`, `predicate-matrix-binary.txt`
and `JPEGImages` listed, or step 5 will raise `FileNotFoundError`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, glob, shutil, pathlib
assert os.path.exists(AWA2_ZIP), f"Zip not found at {AWA2_ZIP} — fix AWA2_ZIP in the Config cell."

# Unzip into a staging dir, then normalize: locate classes.txt wherever it landed and make
# ./Animals_with_Attributes2/ point at its parent. Handles both zip layouts — the folder at
# the top level, or the folder's *contents* at the top level.
!rm -rf _awa2_stage Animals_with_Attributes2
!mkdir -p _awa2_stage
!unzip -q -o "$AWA2_ZIP" -d _awa2_stage

hits = glob.glob("_awa2_stage/**/classes.txt", recursive=True)
assert hits, "classes.txt not found inside the zip — is this the Animals_with_Attributes2 archive?"
src_root = str(pathlib.Path(hits[0]).parent)
assert os.path.isdir(os.path.join(src_root, "JPEGImages")), \
    f"Found classes.txt at {src_root} but no JPEGImages/ beside it."
shutil.move(src_root, "Animals_with_Attributes2")
!rm -rf _awa2_stage

print("\nContents of Animals_with_Attributes2 (must show classes.txt + JPEGImages):")
!ls Animals_with_Attributes2

## 5. Build the frozen index, split, and metadata caches

`data_loader.py` freezes the image index (index→file contract), the stratified 80/20
split, and the small json metadata caches. Fast, CPU-only.

In [ ]:
!python src/awa2/data_loader.py

## 6. Extract CLIP image features (both splits) + the prompt bank

`clip_features.py` encodes all ~37k images on the GPU (resume-checkpointed every 2048),
L2-normalizes, and verifies row-count alignment + unit norms before saving each split.
`clip_prompts.py` builds the `[85, n, 512]` per-predicate prompt bank CLAY needs.

In [ ]:
!python src/awa2/clip_features.py
!python src/awa2/clip_prompts.py

## 7. Save the outputs

Copies every downstream artifact to Drive (so it persists) **and** triggers browser
downloads. Drop these into your VM's / local `Deep-Learning-Project/artifacts/awa2/`
folder — then the AwA2 tiers run without the raw dataset present.

In [ ]:
import os, shutil
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

outputs = [
    "artifacts/awa2/clip_image_features_train.pt",
    "artifacts/awa2/clip_image_features_test.pt",
    "artifacts/awa2/clip_attr_prompt_bank.pt",
    "artifacts/awa2/image_index.json",
    "artifacts/awa2/split_train_idx.pt",
    "artifacts/awa2/split_test_idx.pt",
    "artifacts/awa2/class_names.json",
    "artifacts/awa2/predicate_names.json",
    "artifacts/awa2/class_attribute_matrix.json",
]
for f in outputs:
    assert os.path.exists(f), f"Missing {f} — did steps 5–6 finish without errors?"
    shutil.copy(f, DRIVE_OUT_DIR)
    print("copied to Drive:", f)

from google.colab import files
for f in outputs:
    files.download(f)